# Regression Using Pytorch

In [23]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
import os
os.chdir("D:\Machine Learning\Datasets")

In [24]:
import torch 
import torch.nn as nn
from torchsummary import summary

In [25]:
#  sklearn imports 
from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, log_loss, r2_score

In [44]:
housing = pd.read_csv("Housing.csv")
dum_hous = pd.get_dummies(housing, drop_first= True) # for one hot encoding all the data
y = dum_hous[['price']]
X = dum_hous.drop('price', axis = 1)

In [27]:
scaler_x,scaler_y = MinMaxScaler(), MinMaxScaler()
X_train, X_test, y_train, y_test = train_test_split(X,y, test_size=0.3, random_state = 25)
X_scl_trn = scaler_x.fit_transform(X_train)
X_scl_tst = scaler_x.transform(X_test)

In [28]:
y_scl_trn = scaler_y.fit_transform(y_train)
y_scl_tst = scaler_y.transform(y_test)

In [29]:
(X_scl_trn.shape, y_scl_trn.shape), (X_scl_tst.shape, y_scl_tst.shape)

(((382, 11), (382, 1)), ((164, 11), (164, 1)))

In [30]:
X_torch = torch.from_numpy(X_scl_trn)
y_torch = torch.from_numpy(y_scl_trn)
X_torch_test = torch.from_numpy(X_scl_tst)
y_torch_test = torch.from_numpy(y_scl_tst)
print(X_torch.size())
print(y_torch.size())
print(X_torch_test.size())
print(y_torch_test.size())

torch.Size([382, 11])
torch.Size([382, 1])
torch.Size([164, 11])
torch.Size([164, 1])


### Model Defination

In [ ]:
torch.manual_seed(25)
model = nn.Sequential(nn.Linear(in_features=X_scl_trn.shape[1], out_features=7),
                    nn.Linear(in_features=7, out_features=4),
                    nn.ReLU()
                    ,nn.Linear(in_features=4, out_features=1)
                    )

In [32]:
summary(model, (1,X_scl_trn.shape[1]))

----------------------------------------------------------------
        Layer (type)               Output Shape         Param #
            Linear-1                 [-1, 1, 7]              84
            Linear-2                 [-1, 1, 4]              32
              ReLU-3                 [-1, 1, 4]               0
            Linear-4                 [-1, 1, 1]               5
Total params: 121
Trainable params: 121
Non-trainable params: 0
----------------------------------------------------------------
Input size (MB): 0.00
Forward/backward pass size (MB): 0.00
Params size (MB): 0.00
Estimated Total Size (MB): 0.00
----------------------------------------------------------------


In [33]:
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr = 0.1)
optimizer

Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.1
    maximize: False
    weight_decay: 0
)

#### Generating Predictions with default weights

In [34]:
y_pred = model(X_torch.float())
y_pred[:3]

tensor([[-0.0705],
        [-0.1197],
        [-0.0852]], grad_fn=<SliceBackward0>)

In [35]:
print(y_torch.shape)

torch.Size([382, 1])


#### Initial MSE

In [36]:
loss = criterion(y_pred, y_torch.float())
loss

tensor(0.1724, grad_fn=<MseLossBackward0>)

In [37]:
for epoch in np.arange(0,1000):
    # Forward Pass:  Compute predicted y by passing x to the model
    y_pred_prob = model(X_torch.float()) # Inferencing of X
    # Compute and print loss
    loss = criterion(y_pred_prob, y_torch.float())
    if epoch % 100 == 0:
        print('Epoch', epoch+1, 'Loss:', loss.item())
    # Zero Gradients, perform a backward pass, and update the weights.
    optimizer.zero_grad() # all gradients are equaled to 0
    #perform a backward pass (backpropogation)
    loss.backward()
    #
    optimizer.step()


Epoch 1 Loss: 0.17235533893108368
Epoch 101 Loss: 0.007769593503326178
Epoch 201 Loss: 0.007756020408123732
Epoch 301 Loss: 0.00778915174305439
Epoch 401 Loss: 0.007813530042767525
Epoch 501 Loss: 0.0078031267039477825
Epoch 601 Loss: 0.0077705481089651585
Epoch 701 Loss: 0.007847209461033344
Epoch 801 Loss: 0.007860979065299034
Epoch 901 Loss: 0.007878964766860008


In [38]:
loss


tensor(0.0077, grad_fn=<MseLossBackward0>)

In [39]:
X_torch_test = torch.from_numpy(X_scl_tst)
type(X_torch_test)

torch.Tensor

In [40]:
y_pred_scl = model(X_torch_test.float())
type(y_pred_scl)

torch.Tensor

In [41]:
y_pred_scl = y_pred_scl.detach().numpy()
y_pred_scl[:3]

array([[0.3157855 ],
       [0.11802845],
       [0.16049589]], dtype=float32)

In [42]:
y_pred = scaler_y.inverse_transform(y_pred_scl)
y_pred.shape

(164, 1)

### Test Set for r2 score

In [43]:
r2_score(y_test,y_pred)

0.6249430775642395